# 🔬 Week 4: Hyperparameter Tuning & Model Explainability
## Telco Customer Churn — Optuna Tuning + SHAP Analysis

**Goal:** Push model performance beyond the baseline and understand *why* the model makes each prediction.

### Agenda
1. Load processed data
2. Run Optuna hyperparameter search (50 trials)
3. Compare tuned vs baseline XGBoost
4. SHAP global explainability (summary + bar)
5. SHAP local explainability (waterfall for specific customers)
6. Log tuned model to MLflow

---
## 0. Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.pipeline import run_full_pipeline, NUMERICAL_FEATURES, CATEGORICAL_FEATURES
from src.models.train import train_xgboost, DEFAULT_XGB_PARAMS
from src.models.tune import run_study, get_best_model, log_tuned_model, plot_optimization_history, plot_param_importances
from src.models.evaluate import compute_metrics, print_report
from src.models.explain import compute_shap_values, plot_summary, plot_bar, plot_waterfall, top_features, get_feature_names

sns.set_theme(style='darkgrid', palette='deep')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.float_format', '{:.4f}'.format)

print('Setup complete ✓')

---
## 1. Load Data

In [ ]:
X_train, X_test, y_train, y_test, pipe = run_full_pipeline()
feature_names = get_feature_names(pipe)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'Features: {len(feature_names)}')

---
## 2. Baseline XGBoost (for comparison)

In [ ]:
baseline = train_xgboost(X_train, y_train, X_test, y_test, log_to_mlflow=False)
baseline_metrics = baseline['metrics']
print('Baseline XGBoost:')
for k, v in baseline_metrics.items():
    print(f'  {k:<12} {v:.4f}')

---
## 3. Optuna Hyperparameter Search

This will run **50 trials** of 5-fold cross-validation (≈ 5 minutes).  
Each trial explores a different combination of `n_estimators`, `learning_rate`, `max_depth`, etc.

In [ ]:
study = run_study(X_train, y_train, n_trials=50, show_progress=True)

print(f'\nBest CV AUC:   {study.best_value:.4f}')
print(f'Best params:')
for k, v in study.best_params.items():
    print(f'  {k:<25} {v}')

### Optimisation History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_optimization_history(study, ax=axes[0])
plot_param_importances(study, ax=axes[1])
plt.tight_layout()
plt.show()

---
## 4. Train & Evaluate Tuned Model

In [ ]:
tuned_model = get_best_model(study, X_train, y_train)
tuned_metrics = compute_metrics(tuned_model, X_test, y_test)

# Side-by-side comparison
comparison = pd.DataFrame({
    'Baseline XGBoost': baseline_metrics,
    'Tuned XGBoost':    tuned_metrics,
}).T

display(comparison.style
    .background_gradient(cmap='RdYlGn', axis=0)
    .format('{:.4f}')
    .set_caption('Baseline vs Tuned XGBoost'))

In [ ]:
# Delta
delta = pd.Series(tuned_metrics) - pd.Series(baseline_metrics)
print('Improvement (tuned - baseline):')
for k, v in delta.items():
    arrow = '▲' if v > 0 else '▼'
    print(f'  {arrow} {k:<12}  {v:+.4f}')

In [ ]:
print_report(tuned_model, X_test, y_test, model_name='Tuned XGBoost')

---
## 5. SHAP Global Explainability

SHAP tells us **which features push predictions toward churn or away from it**, and by how much.

In [ ]:
shap_values = compute_shap_values(tuned_model, X_test)
print(f'SHAP values shape: {shap_values.values.shape}')

### 5a. Summary Beeswarm Plot
Each dot = one customer. Red = high feature value, blue = low. Position on x-axis = impact on churn probability.

In [ ]:
plot_summary(shap_values, feature_names=feature_names, max_display=20, save=True)

### 5b. Bar Plot — Mean |SHAP| (Global Feature Importance)

In [ ]:
plot_bar(shap_values, feature_names=feature_names, max_display=20, save=True)

### 5c. Top 10 Features Table

In [ ]:
top = top_features(shap_values, feature_names=feature_names, n=10)
top_df = pd.DataFrame(top, columns=['Feature', 'Mean |SHAP|'])
top_df.index += 1
display(top_df.style
    .background_gradient(cmap='YlOrRd', subset=['Mean |SHAP|'])
    .format({'Mean |SHAP|': '{:.4f}'})
    .set_caption('Top 10 Features by SHAP Importance'))

---
## 6. SHAP Local Explainability — Individual Predictions

A waterfall plot answers: **"Why did the model say THIS customer would churn?"\ **

In [ ]:
# Find a high-risk customer (predicted churn prob > 0.7)
y_prob = tuned_model.predict_proba(X_test)[:, 1]
high_risk_idx = np.where(y_prob > 0.7)[0]

if len(high_risk_idx) > 0:
    idx = high_risk_idx[0]
    print(f'Customer #{idx}: predicted churn probability = {y_prob[idx]:.1%}')
    print(f'Actual label: {"Churned" if y_test.iloc[idx] == 1 else "Stayed"}')
    plot_waterfall(shap_values, sample_idx=idx, feature_names=feature_names, save=True)
else:
    print('No customers with >70% predicted churn in test set — using index 0')
    plot_waterfall(shap_values, sample_idx=0, feature_names=feature_names, save=True)

In [ ]:
# Also show a low-risk customer for contrast
low_risk_idx = np.where(y_prob < 0.1)[0]
if len(low_risk_idx) > 0:
    idx = low_risk_idx[0]
    print(f'Customer #{idx}: predicted churn probability = {y_prob[idx]:.1%}')
    print(f'Actual label: {"Churned" if y_test.iloc[idx] == 1 else "Stayed"}')
    plot_waterfall(shap_values, sample_idx=idx, feature_names=feature_names, save=True)

---
## 7. Log Tuned Model to MLflow

In [ ]:
run_id = log_tuned_model(tuned_model, study, X_test, y_test)
print(f'MLflow run_id: {run_id}')
print('Open http://localhost:5000 to view the tuned run alongside baselines')

---
## ✅ Week 4 Summary

| What | Result |
|------|--------|
| Optuna trials | 50 (5-fold CV each) |
| Best CV AUC | See `study.best_value` above |
| SHAP artefacts | `models/artefacts/shap_summary.png`, `shap_bar.png`, `shap_waterfall_*.png` |
| MLflow run | Logged as `XGBoost_Tuned` in `churn-baseline` experiment |

### Key Insight Pattern (typical for Telco churn)
- **Tenure** — short-tenure customers are highest risk
- **Contract type** — Month-to-month contracts churn significantly more
- **Total charges / CLV** — lower overall spend = higher churn risk
- **Internet service** — Fiber optic customers churn at higher rates (possibly due to cost)

**Next: Week 5 — FastAPI REST API + Docker containerisation**
- Build a `/predict` endpoint that accepts customer JSON and returns churn probability
- Package everything in a Docker container
- Write API integration tests